In [12]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch1. 허깅페이스 모델 사용</font>
- pipeline(): 모델을 로컬에 다운 후 결과를 출력하는 로컬 사용 방식
- Inference API: HTTP 요청으로 결과만 받아오는 방식



        - 처음 모델 사용 시 "C:/Users/Admin/.cache/huggingface" 다운로드로 시간 소요 발생
        
|**Task**|**설명**|
|:---|:---|
|text-classification (sentiment-analysis) | 감정 분석, 뉴스 분류, 리뷰 분류 등 문장 분류|
|zero-shot-classification | 레이블에 대한 별도 학습 없이 후보 레이블 중에서 분류|
|text-generation | GPT 계열 모델을 이용한 텍스트 생성|
|fill-mask | 문장 안의 빈칸(마스크)에 들어갈 단어 예측|
|ner (token-classification) | 개체명 인식(사람, 조직, 장소 등 라벨링)|
|question-answering | 주어진 지문(context)을 근거로 질문에 답변|
|summarization | 긴 문서를 짧게 요약|
|translation | 서로 다른 언어 간 번역|
|image-to-text | 이미지 내용을 설명하는 문장 생성|
|image-classification | 이미지가 어떤 대상인지 분류|



In [13]:
import warnings
import os
import logging
 
# 경고 메시지 제거
warnings.filterwarnings('ignore')
 
# transformers 라이브러리의 로깅 레벨을 ERROR로 조정 (경고 숨김)
logging.getLogger("transformers").setLevel(logging.ERROR)
 
# Hugging Face 캐시 관련 symlink 경고 제거
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# 1. TEXT 기반 감정 분석
- 감성 분석(Sentiment Analysis) : 텍스트에 들어있는 작성자의 태도, 의견, 감정 상태(긍정, 부정, 중립 등)를 컴퓨터가 자동으로 판별해내는 자연어 처리(NLP) 기술


    - pipeline() : 토큰화 → 워드 임베딩 → 모델 → 예측

In [14]:
from transformers import pipeline
classifier = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english")
classifier("I've been waiting for a Hugging face course my whole life")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982088804244995}]

In [17]:
# 모델의 파라미터와 용량
from transformers import AutoModel
model = AutoModel.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

# 전체 파라미터 수
total_params = sum(p.numel() for p in model.parameters())

In [19]:
print(f"전체 파라미터 수: {total_params:,}")
print(f"전체 파라미터 용량: {total_params/1024/1024:.3f}MB")

전체 파라미터 수: 66,362,880
전체 파라미터 용량: 63.289MB


In [20]:
from transformers import pipeline
classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english")

# 다항의 경우 list
classifier([
    "I've been waiting for a Hugging face course my whole life",
    "I hate this"
])

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982088804244995},
 {'label': 'NEGATIVE', 'score': 0.9996224641799927}]

In [21]:
classifier([
    "I've been waiting for a Hugging face course my whole life",
    "이 영화는 재미 없었어요" # 영어로 학습한 모델 : 한국어에 대한 정확도는 낮은 편
]) 

[{'label': 'POSITIVE', 'score': 0.9982088804244995},
 {'label': 'POSITIVE', 'score': 0.879510760307312}]

In [22]:
classifier = pipeline(
    task="sentiment-analysis",
    model="daekeun-ml/koelectra-small-v3-nsmc")
text = ['힘들어요','오늘 기분 좋은데','너 싫어','내가 최고야']
classifier(text)

Device set to use cpu


[{'label': '0', 'score': 0.9957089424133301},
 {'label': '1', 'score': 0.9966667294502258},
 {'label': '0', 'score': 0.9964488744735718},
 {'label': '1', 'score': 0.9213551878929138}]

In [23]:
for text, result in zip(text, classifier(text)):
    label = "긍정" if result['label']=='1' else '부정'
    print(f"'{text}' → {label} {result['score']:.2%}")

'힘들어요' → 부정 99.57%
'오늘 기분 좋은데' → 긍정 99.67%
'너 싫어' → 부정 99.64%
'내가 최고야' → 긍정 92.14%


# 2. Zero-shot 분류
- Zero-shot 분류 : 사전 학습 과정에서 본 적 없는 새로운 범주의 데이터를 추가 학습이나 훈련 데이터(Zero-shot = 0개의 학습 데이터) 없이 분류해내는 머신러닝 기법

In [ ]:
classifier = pipeline(
    task='zero-shot-classification',
    model='facebook/bart-large-mnli')

classifier(
    "I have a problem with my i-phone that needs to be resolved asap!!",
    candidate_labels=['phone', 'urgent', 'tablet', 'computer'] # 주제 제시
          )

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


In [ ]:
classifier(
    "This is a course about the Tranformers library",
    candidate_labels=['education','business','phone']
)